In [ ]:


## Packages ---
import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
from datetime import datetime
import time
import functools as ft
from IPython.display import display
pd.set_option('display.max_columns', None)


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'Census'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_csm = path_prod / 'Chamber Study Missions' / date.today().strftime('%Y') / 'Hispanic Chamber of Commerce'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())


export=False



************************************************************************************************

Table 1

************************************************************************************************

Chamber_H_1
- Table 1A

In [ ]:


path_in  = path_csm / 'original_exports'
indicators = ['Chamber_H_1aa', 'Chamber_H_1ab']
geographies = ['Counties', 'MPO']
table_id = '1A'

list_df_geo = []

for geography in geographies:
    list_df_ind = []

    for indicator in indicators:
        if indicator == 'Chamber_H_1aa':
            estimate = 'ACS5'
        if indicator == 'Chamber_H_1ab':
            estimate = 'SUBJECT5'
        workbook = f'{indicator} {geography} {estimate}.xlsx'
        sheet_name = geography

        if geography == 'Counties':
            geo_ID = ['County Name']
        if geography == 'MPO':
            geo_ID = ['MPO']


        ## Import ---

        file_in = path_in / workbook
        df_census = pd.read_excel(file_in, sheet_name=sheet_name)

        df_census = df_census[geo_ID + ['Year', 'Variable', 'Population']]
        df_census = df_census.pivot_table(index = geo_ID + ['Year']
                                            , columns = 'Variable'
                                            , values = 'Population').reset_index()
        df_census = df_census.sort_values(geo_ID + ['Year'], ascending = [item in geo_ID for item in geo_ID] + [False])
        df_census = df_census[geo_ID + ['Year', 'Under 18', 'Adult']]
        df_census = df_census.rename(columns = {'Under 18':f'Under 18_{indicator}', 'Adult':f'Adult_{indicator}'})
        list_df_ind.append(df_census)

    df_census = ft.reduce(lambda left, right: pd.merge(left, right, on = geo_ID + ['Year'], how = 'outer'), list_df_ind)
    df_census = df_census.sort_values(['Year'] + geo_ID, ascending=[False, True])
    df_census['Under 18_Chamber_H_1ab'] = df_census['Under 18_Chamber_H_1ab'] - df_census['Under 18_Chamber_H_1aa']
    df_census['Adult_Chamber_H_1ab'   ] = df_census['Adult_Chamber_H_1ab'   ] - df_census['Adult_Chamber_H_1aa'   ]
    indicator = indicator[:-2]
    df_census = df_census.rename(columns={'County Name':'Geography', 'MPO':'Geography'})
    list_df_geo.append(df_census)

df_census = pd.concat(list_df_geo)


df_census['Sort'] = pd.Categorical(df_census['Geography'], ['El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                    ])
df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
df_census = df_census.drop('Sort', axis=1)
df_census = df_census.reset_index(drop=True)

df_census.columns = ['Geography', 'Year', 'Hispanic Under 18', 'Hispanic Adult', 'Non-Hispanic Under 18', 'Non-Hispanic Adult']
display(df_census)


## Exporting ---

if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_census.to_excel(writer, sheet_name=table_id, index=False)

        

Chamber_H_2
- Table 1B
- Table 1D


In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
geographies =  'MPO'
indicator = 'Chamber_H_2'
geography = 'MPO'
table_id = '1B'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_census = pd.read_excel(file_in, sheet_name=sheet_name)

df_census = df_census[['MPO', 'Year', 'Variable', 'Population']]
df_census.loc[ df_census['Variable'].str.contains('Under 18'), 'Age Group'] = 'Children (under 18)'
df_census.loc[~df_census['Variable'].str.contains('Under 18'), 'Age Group'] = 'Adults (18+)'

df_census.loc[ df_census['Variable'].str.contains('born|Born'), 'Category'] = 'A'
df_census.loc[~df_census['Variable'].str.contains('born|Born'), 'Category'] = 'B'



def re_remove_post(x, exp = '('):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0].strip()
    
df_census['Variable'] = df_census['Variable'].apply(re_remove_post)


df_census['Percentage'] = df_census['Population'] / df_census.groupby(['MPO', 'Year', 'Category', 'Age Group'])['Population'].transform('sum')

                

df_census = df_census.sort_values(['Year', 'Category', 'Variable'], ascending=[False, False, False])


df_census1 = df_census.pivot_table(index = ['MPO', 'Year', 'Variable']
                                    , columns = 'Age Group'
                                    , values = 'Percentage').reset_index()
df_census1 = df_census1.rename(columns={'Adults (18+)':'Adults (18+)_pct', 'Children (under 18)':'Children (under 18)_pct'})
df_census2 = df_census.pivot_table(index = ['MPO', 'Year', 'Variable']
                                    , columns = 'Age Group'
                                    , values = 'Population').reset_index()
df_census = df_census1.merge(df_census2, on=['MPO', 'Year', 'Variable'])


df_census['Sort'] = pd.Categorical(df_census['Variable'], ['Born in the US'
                                                            , 'Not born in the US'
                                                            , 'US citizens'
                                                            , 'Not US citizens'
                                                        ])
df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
df_census = df_census.drop('Sort', axis=1)
df_census = df_census.reset_index(drop=True)


display(df_census)



## Exporting ---

if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_census.to_excel(writer, sheet_name=table_id, index=False)



In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
geographies =  'Counties'
indicator = 'Chamber_H_2'
geography = 'Counties'
table_id = '1D'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_counties = pd.read_excel(file_in, sheet_name=sheet_name)

df_counties = df_counties[['County Name', 'Year', 'Variable', 'Population']]
df_counties.loc[ df_counties['Variable'].str.contains('Under 18'), 'Age Group'] = 'Children (under 18)'
df_counties.loc[~df_counties['Variable'].str.contains('Under 18'), 'Age Group'] = 'Adults (18+)'

df_counties.loc[ df_counties['Variable'].str.contains('born|Born'), 'Category'] = 'A'
df_counties.loc[~df_counties['Variable'].str.contains('born|Born'), 'Category'] = 'B'



def re_remove_post(x, exp = '('):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0].strip()
    
df_counties['Variable'] = df_counties['Variable'].apply(re_remove_post)


df_counties['Percentage'] = df_counties['Population'] / df_counties.groupby(['County Name', 'Year', 'Category', 'Age Group'])['Population'].transform('sum')

                

df_counties = df_counties.sort_values(['Year', 'County Name', 'Category', 'Variable'], ascending=[False, True, False, False])


df_counties1 = df_counties.pivot_table(index = ['County Name', 'Year', 'Variable']
                                        , columns = 'Age Group'
                                        , values = 'Percentage').reset_index()
df_counties1 = df_counties1.rename(columns={'Adults (18+)':'Adults (18+)_pct', 'Children (under 18)':'Children (under 18)_pct'})
df_counties2 = df_counties.pivot_table(index = ['County Name', 'Year', 'Variable']
                                        , columns = 'Age Group'
                                        , values = 'Population').reset_index()
df_counties = df_counties1.merge(df_counties2, on=['County Name', 'Year', 'Variable'])


df_counties['Sort'] = pd.Categorical(df_counties['Variable'], ['Born in the US'
                                                                , 'Not born in the US'
                                                                , 'US citizens'
                                                                , 'Not US citizens'
                                                            ])
df_counties = df_counties.sort_values(['Year', 'County Name', 'Sort'], ascending=[False, True, True])
df_counties = df_counties.drop('Sort', axis=1)
df_counties = df_counties.reset_index(drop=True)






df_mpo = df_census.copy()
df_mpo      = df_mpo     .rename(columns={'MPO'        :'Geography'})
df_counties = df_counties.rename(columns={'County Name':'Geography'})

df_mpo = pd.concat([df_counties, df_mpo])

df_mpo['Population'] =  df_mpo['Adults (18+)'] + df_mpo['Children (under 18)']
df_mpo = df_mpo[['Geography', 'Variable', 'Year', 'Population']]

df_mpo = df_mpo.pivot_table(index = ['Geography', 'Year']
                            , columns = 'Variable'
                            , values = 'Population').reset_index()

df_mpo = df_mpo[['Geography', 'Year', 'Born in the US', 'Not born in the US']]
df_mpo['Total'] = df_mpo['Born in the US'] + df_mpo['Not born in the US']
df_mpo['Born in the US_pct'    ] = df_mpo['Born in the US'    ]/df_mpo['Total']
df_mpo['Not born in the US_pct'] = df_mpo['Not born in the US']/df_mpo['Total']

df_mpo = df_mpo[['Geography', 'Year', 'Born in the US', 'Born in the US_pct', 'Not born in the US', 'Not born in the US_pct']]


df_mpo['Sort'] = pd.Categorical(df_mpo['Geography'], ['El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                    ])
df_mpo = df_mpo.sort_values(['Year', 'Sort'], ascending=[False, True])
df_mpo = df_mpo.drop('Sort', axis=1)
df_mpo = df_mpo.reset_index(drop=True)

display(df_mpo)





## Exporting ---

if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_mpo.to_excel(writer, sheet_name=table_id, index=False)



In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
geographies =  'MPO'
indicator = 'Chamber_H_4'
geography = 'MPO'
table_id = '1E'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography


## Import ---

file_in = path_in / workbook
df_acs = pd.read_excel(file_in, sheet_name=sheet_name)


display(df_acs)


## Exporting ---

if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_acs.to_excel(writer, sheet_name=table_id, index=False)



In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'PUMS5'
geographies =  'MPO'
indicator = 'Chamber_H_4'
geography = 'MPO'
table_id = '1E'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography


## Import ---

file_in = path_in / workbook
df_pums = pd.read_excel(file_in, sheet_name=sheet_name)
df_pums[df_pums['Year'] == 2021]

df_pums = df_pums[['Year', 'HISP', 'Population']]


df_pums['Sort'] = pd.Categorical(df_pums['HISP'], [
'Mexican'
, 'Puerto Rican'
, 'Cuban'
, 'Dominican'
, 'Costa Rican'
, 'Guatemalan'
, 'Honduran'
, 'Nicaraguan'
, 'Panamanian'
, 'Salvadoran'
, 'Other Central American'
, 'Argentinean'
, 'Bolivian'
, 'Chilean'
, 'Colombian'
, 'Ecuadorian'
, 'Paraguayan'
, 'Peruvian'
, 'Uruguayan'
, 'Venezuelan'
, 'Other South American'
, 'Spaniard'
, 'All Other Spanish/Hispanic/Latino'
])

df_pums = df_pums.sort_values(['Year', 'Sort'], ascending=[False, True])
df_pums = df_pums.drop('Sort', axis=1)
df_pums = df_pums.reset_index(drop=True)


display(df_pums)


## Exporting ---

if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_pums.to_excel(writer, sheet_name=table_id, index=False)



************************************************************************************************

Table 2

************************************************************************************************

Chamber_H_5
- Table 2A

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
indicator = 'Chamber_H_5'
geography = 'Counties'
table_id = '2A'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_counties = pd.read_excel(file_in, sheet_name=sheet_name)
df_counties = df_counties.rename(columns={'County Name':'Geography'})
df_counties = df_counties.pivot_table(index = ['Geography', 'Year']
                                        , columns = 'Race_Ethnicity'
                                        , values = 'Median Household Income').reset_index()


df_counties

geography = 'MPO'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

file_in = path_in / workbook
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)
df_mpo = df_mpo.rename(columns={'MPO':'Geography'})
df_mpo = df_mpo.pivot_table(index = ['Geography', 'Year']
                                        , columns = 'Race_Ethnicity'
                                        , values = 'Median Household Income').reset_index()


df_census = pd.concat([df_counties, df_mpo])

df_census['Sort'] = pd.Categorical(df_census['Geography'], [
                                                        'El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                   ])

df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
df_census = df_census.drop('Sort', axis=1)
df_census = df_census.reset_index(drop=True)

display(df_census)




## Exporting ---

if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_census.to_excel(writer, sheet_name=table_id, index=False)



In [ ]:


# path_in  = path_csm / 'original_exports'
# estimate = 'ACS1'
# indicator = 'Chamber_H_5'
# geography = 'Counties'
# table_id = '2A'

# workbook = f'{indicator} {geography} {estimate}.xlsx'
# sheet_name = geography

# ## Import ---

# file_in = path_in / workbook
# df_counties = pd.read_excel(file_in, sheet_name=sheet_name)
# df_counties = df_counties.rename(columns={'County Name':'Geography'})
# df_counties = df_counties.pivot_table(index = ['Geography', 'Year']
#                                         , columns = 'Race_Ethnicity'
#                                         , values = 'Median Household Income').reset_index()


# df_counties

# geography = 'MPO'

# workbook = f'{indicator} {geography} {estimate}.xlsx'
# sheet_name = geography

# file_in = path_in / workbook
# df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)
# df_mpo = df_mpo.rename(columns={'MPO':'Geography'})
# df_mpo = df_mpo.pivot_table(index = ['Geography', 'Year']
#                                         , columns = 'Race_Ethnicity'
#                                         , values = 'Median Household Income').reset_index()


# df_census = pd.concat([df_counties, df_mpo])

# df_census['Sort'] = pd.Categorical(df_census['Geography'], [
#                                                         'El Dorado'
#                                                         , 'Placer'
#                                                         , 'Sacramento'
#                                                         , 'Sutter'
#                                                         , 'Yolo'
#                                                         , 'Yuba'
#                                                         , 'SACOG'
#                                                    ])

# df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
# df_census = df_census.drop('Sort', axis=1)
# df_census = df_census.reset_index(drop=True)

# display(df_census)




# ## Exporting ---

# if export:
#     workbook = 'post2.xlsx'
#     file_out = path_csm / 'post' / workbook

#     with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#         df_census.to_excel(writer, sheet_name=table_id, index=False)



Chamber_H_6
- Table 2B

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'PUMS5'
indicator = 'Chamber_H_6'
geography = 'Counties'
table_id = '2B'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = 'PUMA'

## Import ---

file_in = path_in / workbook
df_counties = pd.read_excel(file_in, sheet_name=sheet_name)
df_counties = df_counties.rename(columns={'County Name':'Geography'})
df_counties['GRPIP'] = df_counties['GRPIP'].astype(int)
df_counties = df_counties[df_counties['GRPIP'] != 0]
wm = lambda x: np.average(x, weights = df_counties.loc[x.index, "Households"])/100 # weighted average
df_counties = df_counties.groupby(['Geography', 'Year', 'HISP'], as_index=False).agg(GRPIP = ('GRPIP', wm))

df_counties = df_counties.pivot_table(index = ['Geography', 'Year']
                                        , columns = 'HISP'
                                        , values = 'GRPIP').reset_index()



geography = 'MPO'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

file_in = path_in / workbook
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)
df_mpo = df_mpo.rename(columns={'MPO':'Geography'})
df_mpo['GRPIP'] = df_mpo['GRPIP'].astype(int)
df_mpo = df_mpo[df_mpo['GRPIP'] != 0]
wm = lambda x: np.average(x, weights = df_mpo.loc[x.index, "Households"])/100 # weighted average
df_mpo = df_mpo.groupby(['Geography', 'Year', 'HISP'], as_index=False).agg(GRPIP = ('GRPIP', wm))

df_mpo = df_mpo.pivot_table(index = ['Geography', 'Year']
                                        , columns = 'HISP'
                                        , values = 'GRPIP').reset_index()


df_census = pd.concat([df_counties, df_mpo])

df_census['Sort'] = pd.Categorical(df_census['Geography'], [
                                                        'El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                    ])
df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
df_census = df_census.drop('Sort', axis=1)
df_census = df_census.reset_index(drop=True)

display(df_census)




## Exporting ---

if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_census.to_excel(writer, sheet_name=table_id, index=False)



Chamber_H_7
- Table 2C

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS1'
indicator = 'Chamber_H_7'
geography = 'MPO'
table_id = '2C'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

file_in = path_in / workbook
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)
df_mpo = df_mpo.rename(columns={'MPO':'Geography'})
df_mpo = df_mpo.pivot_table(index=['Geography', 'Year']
                                        , columns='Race_Ethnicity'
                                        , values='Median Household Income').reset_index()

df_mpo = df_mpo.sort_values(['Year'], ascending=[True])
df_mpo = df_mpo[df_mpo['Year'] >= 2010]
df_mpo = df_mpo.reset_index(drop=True)

display(df_mpo)


## Exporting ---

if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_mpo.to_excel(writer, sheet_name=table_id, index=False)



In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'PUMS1'
indicator = 'Chamber_H_7'
geography = 'MPO'
table_id = '2C'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography



## Importing ---

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

file_in = path_in / workbook
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)


## Organizing ---

df_mpo['HINCP'] = df_mpo['HINCP'].astype(int)

# wm = lambda x: np.average(x, weights = df_mpo.loc[x.index, "Households"]) # weighted average (or Population or Households)
# df_mpo = df_mpo.groupby(['Year', 'HISP'], as_index=False).agg(HINCP=('HINCP', wm))

df_mpo = df_mpo.sort_values(['Year'], ascending=[True])
df_mpo = df_mpo.reset_index(drop=True)


df_rep = df_mpo.copy()
df_rep = df_rep[['Year', 'HISP', 'HINCP', 'Households']]
df_rep = df_rep[df_rep['Households'] != 0]
df_rep = df_rep.reset_index(drop=True)

list_keys = []
for list_ in df_rep.values:
    list_keys.append(tuple(list_[:-1]))

list_values = []
for list_ in df_rep.values:
    list_values.append(list_[-1])

dict_replicates = dict(zip(list_keys, list_values))

list_df = []
for tuple_ in tqdm(list(dict_replicates.keys())):
    
    row  = list(tuple_)
    wgtp = dict_replicates[tuple_]

    list_df.append(pd.concat([pd.DataFrame(row).T] * wgtp))

df_rep = pd.concat(list_df)
df_rep.columns = ['Year', 'HISP', 'HINCP']

df_rep = df_rep.groupby(['Year', 'HISP'], as_index=False)['HINCP'].median()


df_rep = df_rep.pivot_table(index = ['Year']
                            , columns = 'HISP'
                            , values = 'HINCP').reset_index()


display(df_rep)



************************************************************************************************

Table 3

************************************************************************************************

Chamber_H_8
- Table 3A

In [ ]:


table_id = '3A'


url_edu24 = "https://www3.cde.ca.gov/demo-downloads/acgr/acgr24.txt"
df_edu24 = pd.read_csv(url_edu24, sep = '\t')
# display(df_edu24)


url_edu22 = "https://www3.cde.ca.gov/demo-downloads/acgr/acgr22.txt"
df_edu22 = pd.read_csv(url_edu22, sep = '\t')
# display(df_edu22)

df_edu = pd.concat([df_edu24, df_edu22])
print(df_edu.columns)

df_edu = df_edu[df_edu['ReportingCategory'].isin(['RH', 'RW'])]
df_edu = df_edu[df_edu['AggregateLevel'] == 'C']
df_edu = df_edu[df_edu['CountyName'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]
df_edu['CohortStudents'                      ] = df_edu['CohortStudents'                      ].replace('*', '0').astype(int)
df_edu['Regular HS Diploma Graduates (Count)'] = df_edu['Regular HS Diploma Graduates (Count)'].replace('*', '0').astype(int)
df_edu["Met UC/CSU Grad Req's (Count)"] = df_edu["Met UC/CSU Grad Req's (Count)"].replace('*', '0').astype(int)

df_edu1 = df_edu.groupby(['AcademicYear', 'CountyName', 'ReportingCategory'], as_index=False).agg(TotalStudents=('CohortStudents', 'sum'), Graduated=('Regular HS Diploma Graduates (Count)', 'sum'), Met_UC_req=("Met UC/CSU Grad Req's (Count)", 'sum'))
df_edu2 = df_edu.groupby(['AcademicYear'              , 'ReportingCategory'], as_index=False).agg(TotalStudents=('CohortStudents', 'sum'), Graduated=('Regular HS Diploma Graduates (Count)', 'sum'), Met_UC_req=("Met UC/CSU Grad Req's (Count)", 'sum'))
df_edu2['CountyName']='SACOG'

df_edu = pd.concat([df_edu1, df_edu2])

df_edu['Graduated_pct' ] = df_edu['Graduated' ]/df_edu['TotalStudents']
df_edu['Met_UC_req_pct'] = df_edu['Met_UC_req']/df_edu['Graduated'    ]

df_edu1 = df_edu.pivot_table(index=['AcademicYear', 'CountyName'], columns='ReportingCategory', values='Graduated_pct').reset_index()
df_edu1 = df_edu1.rename(columns={'RH':'Graduating Hispanic_pct', 'RW':'Graduating White_pct'})

df_edu2 = df_edu.pivot_table(index=['AcademicYear', 'CountyName'], columns='ReportingCategory', values='Met_UC_req_pct').reset_index()
df_edu2 = df_edu2.rename(columns={'RH':'Met_UC_req Hispanic_pct', 'RW':'Met_UC_req White_pct'})

df_edu = df_edu1.merge(df_edu2)


df_edu['Sort'] = pd.Categorical(df_edu['CountyName'], ['El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                    ])
df_edu = df_edu.sort_values(['AcademicYear', 'Sort'], ascending=[False, True])
df_edu = df_edu.drop('Sort', axis=1)
df_edu = df_edu.reset_index(drop=True)

display(df_edu)


## Exporting ---

if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_edu.to_excel(writer, sheet_name=table_id, index=False)



Chamber_H_9
- Table 3B

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'PUMS5'
geographies = 'MPO'
indicator = 'Chamber_H_9'
geography = 'MPO'
table_id = '3B'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_edu = pd.read_excel(file_in, sheet_name=sheet_name)
df_edu = df_edu.drop('Percentage', axis=1)
df_edu['AGEP'      ] = df_edu['AGEP'      ].astype(int)
df_edu['Population'] = df_edu['Population'].astype(int)
df_edu

df_edu = df_edu[df_edu['AGEP'] >= 25]

conditions = [
      (df_edu['AGEP'] >= 25) & (df_edu['AGEP'] <= 40)
    , (df_edu['AGEP'] >= 41) & (df_edu['AGEP'] <= 64)
    , (df_edu['AGEP'] >= 65)
    ]

choices = ['Age 25-40', 'Age 41-64', 'Age 65+']

df_edu['Age group'] = np.select(conditions, choices, default='no')

df_edu  = df_edu.groupby(['Year', 'HISP', 'Age group', 'SCHL'], as_index=False).agg(Population = ('Population', 'sum'))
df_edu2 = df_edu.groupby(['Year', 'HISP',              'SCHL'], as_index=False).agg(Population = ('Population', 'sum'))

df_edu2['Age group'] = 'Total'

df_edu = pd.concat([df_edu, df_edu2])

df_edu['Percentage'] = df_edu['Population']/df_edu.groupby(['Year', 'HISP', 'Age group'])['Population'].transform('sum')



df_edu = df_edu.pivot_table(index=['Year', 'Age group', 'SCHL'], columns='HISP', values='Percentage').reset_index()

df_edu['Sort'] = pd.Categorical(df_edu['SCHL'], [
    'Less than HS'
    , 'HS or equivalent'
    , 'Some college/AA'
    , 'BA/BS'
    , 'Graduate'
])

df_edu = df_edu.sort_values(['Year', 'Age group', 'Sort'], ascending=[False, True, True])
df_edu = df_edu.drop('Sort', axis=1)
df_edu = df_edu.reset_index(drop=True)

display(df_edu)


## Exporting ---


if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_edu.to_excel(writer, sheet_name=table_id, index=False)



************************************************************************************************

Table 5

************************************************************************************************

Chamber_H_13
- Table 5A

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
indicator = 'Chamber_H_13'
geography = 'Counties'
table_id = '5A'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_counties = pd.read_excel(file_in, sheet_name=sheet_name)
df_counties = df_counties.rename(columns={'County Name':'Geography'})
df_counties = df_counties.pivot_table(index = ['Geography', 'Year', 'Variable']
                                        , columns = ['Race_Ethnicity']
                                        , values = 'Population').reset_index()


geography = 'MPO'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

file_in = path_in / workbook
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)
df_mpo = df_mpo.rename(columns={'MPO':'Geography'})
df_mpo = df_mpo.pivot_table(index = ['Geography', 'Year', 'Variable']
                                        , columns = ['Race_Ethnicity']
                                        , values = 'Population').reset_index()


df_census = pd.concat([df_counties, df_mpo])

df_census

df_census['Sort'] = pd.Categorical(df_census['Geography'], [
                                                        'El Dorado'
                                                        , 'Placer'
                                                        , 'Sacramento'
                                                        , 'Sutter'
                                                        , 'Yolo'
                                                        , 'Yuba'
                                                        , 'SACOG'
                                                   ])

df_census = df_census.sort_values(['Year', 'Sort'], ascending=[False, True])
df_census = df_census.drop('Sort', axis=1)
df_census = df_census.reset_index(drop=True)

df_census['Not Hispanic'] = df_census['All'] - df_census['Hispanic or Latino']



df_mpo = df_census[df_census['Geography'] == 'SACOG']
df_mpo = df_mpo.reset_index(drop=True)

df_mpo = df_mpo.melt(id_vars=['Geography', 'Year', 'Variable'], var_name = 'Category', value_name='Population')

df_mpo['Percentage'] = df_mpo['Population'] / df_mpo.groupby(['Geography', 'Year', 'Category'])['Population'].transform('sum')

display(df_mpo.head())



## Exporting ---

if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_mpo.to_excel(writer, sheet_name=table_id, index=False)



Chamber_H_14
- Table 5B

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS1'
indicator = 'Chamber_H_14'
geography = 'Counties'
table_id = '5B'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_counties = pd.read_excel(file_in, sheet_name=sheet_name)
df_counties = df_counties.rename(columns={'County Name':'Geography'})
df_counties1 = df_counties[df_counties['Race_Ethnicity'] == 'All'               ]
df_counties2 = df_counties[df_counties['Race_Ethnicity'] == 'Hispanic or Latino']
df_counties1 = df_counties1.pivot_table(index=['Race_Ethnicity', 'Geography'], columns=['Year'], values='Unemployment Rate').reset_index()
df_counties2 = df_counties2.pivot_table(index=['Race_Ethnicity', 'Geography'], columns=['Year'], values='Unemployment Rate').reset_index()
df_counties = pd.concat([df_counties1, df_counties2])


geography = 'MPO'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

file_in = path_in / workbook
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)
df_mpo = df_mpo.rename(columns={'MPO':'Geography'})
df_mpo1 = df_mpo[df_mpo['Race_Ethnicity'] == 'All'               ]
df_mpo2 = df_mpo[df_mpo['Race_Ethnicity'] == 'Hispanic or Latino']
df_mpo1 = df_mpo1.pivot_table(index=['Race_Ethnicity', 'Geography'], columns=['Year'], values='Unemployment Rate').reset_index()
df_mpo2 = df_mpo2.pivot_table(index=['Race_Ethnicity', 'Geography'], columns=['Year'], values='Unemployment Rate').reset_index()
df_mpo = pd.concat([df_mpo1, df_mpo2])


geography = 'States'

workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

file_in = path_in / workbook
df_state = pd.read_excel(file_in, sheet_name=sheet_name)
df_state = df_state.rename(columns={'NAME':'Geography'})
df_state1 = df_state[df_state['Race_Ethnicity'] == 'All'               ]
df_state2 = df_state[df_state['Race_Ethnicity'] == 'Hispanic or Latino']
df_state1 = df_state1.pivot_table(index=['Race_Ethnicity', 'Geography'], columns=['Year'], values='Unemployment Rate').reset_index()
df_state2 = df_state2.pivot_table(index=['Race_Ethnicity', 'Geography'], columns=['Year'], values='Unemployment Rate').reset_index()
df_state = pd.concat([df_state1, df_state2])


df_census = pd.concat([df_counties, df_mpo, df_state])

display(df_census.head())



## Exporting ---

if export:
    workbook = 'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_census.to_excel(writer, sheet_name=table_id, index=False)



Chamber_H_16
- Table 5D

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'PUMS5'
geographies =  'MPO'
indicator = 'Chamber_H_16'
geography = 'MPO'
table_id = '5D'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_pums = pd.read_excel(file_in, sheet_name=sheet_name)
df_pums = df_pums.drop('Percentage', axis=1)


df_pums['Percentage'] = df_pums['Population']/df_pums.groupby(['Year', 'OCCP'])['Population'].transform('sum')

df_pums1 = df_pums.pivot_table(index=['Year', 'OCCP'], columns='HISP', values='Population').reset_index()
df_pums2 = df_pums.pivot_table(index=['Year', 'OCCP'], columns='HISP', values='Percentage').reset_index()
df_pums1 = df_pums1.rename(columns={'Hispanic or Latino':'Hispanic Employed', 'Not Hispanic or Latino':'Not Hispanic or Latino Employed'})
df_pums1['Total Employed'] = df_pums1['Hispanic Employed'] + df_pums1['Not Hispanic or Latino Employed']
df_pums2 = df_pums2.rename(columns={'Hispanic or Latino':'Hispanic or Latino_Percent'})
df_pums2 = df_pums2.drop('Not Hispanic or Latino', axis=1)
df_pums = df_pums1.merge(df_pums2, on=['Year', 'OCCP'])

df_pums['Sort'] = pd.Categorical(df_pums['OCCP'], [
'Architecture and Engineering'
, 'Arts, Design, Entertainment, Sports, and Media'
, 'Building and Grounds Cleaning and Maintenance'
, 'Business Operation Specialists'
, 'Community and Social Services'
, 'Computer and Mathematical'
, 'Construction'
, 'Education, Training, and Library'
, 'Farming, Fishing, and Forestry'
, 'Financial Specialists'
, 'Food Preparation and Serving'
, 'Healthcare Practitioners and Technical'
, 'Healthcare Support'
, 'Installation, Maintenance, and Repair Workers'
, 'Legal'
, 'Life, Physical, and Social Science'
, 'Management, Business, Science, and Arts'
, 'Military Specific'
, 'Office and Administrative Support'
, 'Personal Care and Service'
, 'Production'
, 'Protective Service'
, 'Sales and Related'
, 'Transportation and Material Moving'
])

df_pums = df_pums.sort_values(['Year', 'Sort'], ascending=[False, True])
df_pums = df_pums.drop('Sort', axis=1)
df_pums = df_pums.reset_index(drop=True)

df_pums = df_pums[['Year', 'OCCP', 'Not Hispanic or Latino Employed', 'Hispanic Employed', 'Total Employed', 'Hispanic or Latino_Percent']]

display(df_pums)
print(df_pums[df_pums['Year'] == 2021]['Not Hispanic or Latino Employed'].sum())


## Exporting ---

if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_pums.to_excel(writer, sheet_name=table_id, index=False)



************************************************************************************************

Table 4

************************************************************************************************

Chamber_H_11
- Table 4A

In [ ]:


# SMARTPHONE	Smart phone ownership?
# ACCESSINET	Access to internet	100%-No
# FLAPTOPP	laptop or desktop


path_in  = path_csm / 'original_exports'
estimate = 'PUMS5'
geographies =  'MPO'
indicator = 'Chamber_H_11'
geography = 'MPO'
table_id = '4A'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_pums = pd.read_excel(file_in, sheet_name=sheet_name)
df_pums = df_pums.drop('Percentage', axis=1)

df_pums1 = df_pums.groupby(['Year', 'HISP', 'ACCESSINET'], as_index=False)['Households'].sum()
df_pums1['Percentage'] = 1 - df_pums1['Households']/df_pums1.groupby(['Year', 'HISP'])['Households'].transform('sum')
df_pums1 = df_pums1[df_pums1['ACCESSINET'].str.contains('No access')]
df_pums1 = df_pums1.pivot_table(index=['Year', 'ACCESSINET'], columns='HISP', values='Percentage').reset_index()
df_pums1.columns = ['Year', 'ACCESSINET', 'Access_Hispanic', 'Access_Not Hispanic']
df_pums1 = df_pums1[['Year', 'Access_Hispanic', 'Access_Not Hispanic']]

df_pums2 = df_pums.groupby(['Year', 'HISP', 'SMARTPHONE'], as_index=False)['Households'].sum()
df_pums2['Percentage'] = df_pums2['Households']/df_pums2.groupby(['Year', 'HISP'])['Households'].transform('sum')
df_pums2 = df_pums2[df_pums2['SMARTPHONE'].str.contains('Yes')]
df_pums2 = df_pums2.pivot_table(index=['Year', 'SMARTPHONE'], columns='HISP', values='Percentage').reset_index()
df_pums2.columns = ['Year', 'SMARTPHONE', 'Smartphone_Hispanic', 'Smartphone_Not Hispanic']
df_pums2 = df_pums2[['Year', 'Smartphone_Hispanic', 'Smartphone_Not Hispanic']]

df_pums3 = df_pums.groupby(['Year', 'HISP', 'LAPTOP'], as_index=False)['Households'].sum()
df_pums3['Percentage'] = df_pums3['Households']/df_pums3.groupby(['Year', 'HISP'])['Households'].transform('sum')
df_pums3 = df_pums3[df_pums3['LAPTOP'].str.contains('Yes')]
df_pums3 = df_pums3.pivot_table(index=['Year', 'LAPTOP'], columns='HISP', values='Percentage').reset_index()
df_pums3.columns = ['Year', 'LAPTOP', 'Laptop_Hispanic', 'Laptop_Not Hispanic']
df_pums3 = df_pums3[['Year', 'Laptop_Hispanic', 'Laptop_Not Hispanic']]


df_pums4 = df_pums.groupby(['Year', 'HISP', 'SMARTPHONE', 'LAPTOP'], as_index=False)['Households'].sum()
df_pums4['Percentage'] = df_pums4['Households']/df_pums4.groupby(['Year', 'HISP'])['Households'].transform('sum')
df_pums4 = df_pums4[(df_pums4['SMARTPHONE'] == 'No') & (df_pums4['LAPTOP'] == 'No')]
df_pums4 = df_pums4.pivot_table(index=['Year', 'SMARTPHONE', 'LAPTOP'], columns='HISP', values='Percentage').reset_index()
df_pums4.columns = ['Year', 'SMARTPHONE', 'LAPTOP', 'No Comp or Phone_Hispanic', 'No Comp or Phone_Not Hispanic']
df_pums4 = df_pums4[['Year', 'No Comp or Phone_Hispanic', 'No Comp or Phone_Not Hispanic']]


df_pums = df_pums1.merge(df_pums2.merge(df_pums3.merge(df_pums4)))
df_pums = df_pums.sort_values('Year', ascending=False)
df_pums = df_pums.reset_index(drop=True)

df_pums = df_pums[['Year', 'Access_Hispanic', 'Smartphone_Hispanic', 'Laptop_Hispanic', 'No Comp or Phone_Hispanic', 'Access_Not Hispanic', 'Smartphone_Not Hispanic', 'Laptop_Not Hispanic', 'No Comp or Phone_Not Hispanic']]


display(df_pums)


## Exporting ---

if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_pums.to_excel(writer, sheet_name=table_id, index=False)



************************************************************************************************

Table 6

************************************************************************************************

Chamber_H_17
- Table 6A

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'PUMS5'
geographies =  'MPO'
indicator = 'Chamber_H_17'
geography = 'MPO'
table_id = '6A'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_pums = pd.read_excel(file_in, sheet_name=sheet_name)


df_pums = df_pums[df_pums['MULTG'].str.contains('Yes')]

df_pums = df_pums.reset_index(drop=True)
df_pums = df_pums[['Year', 'HISP', 'Percentage']]

display(df_pums)


## Exporting ---

if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_pums.to_excel(writer, sheet_name=table_id, index=False)



Chamber_H_18
- Table 6B

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'SUBJECT5'
geographies =  'MPO'
indicator = 'Chamber_H_18'
geography = 'MPO'
table_id = '6B'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_pums = pd.read_excel(file_in, sheet_name=sheet_name)
df_pums = df_pums[['Year', 'Race_Ethnicity', 'Variable', 'Households']]
df_pums = df_pums.pivot_table(index=['Year', 'Variable'], columns='Race_Ethnicity', values='Households').reset_index()
df_pums['Not Hispanic or Latino'] = df_pums['All'] - df_pums['Hispanic or Latino']
df_pums = df_pums.melt(id_vars=['Year', 'Variable'], var_name='Race_Ethnicity', value_name='Households')
df_pums = df_pums[df_pums['Race_Ethnicity'] != 'All']
df_pums = df_pums.pivot_table(index=['Year', 'Race_Ethnicity'], columns='Variable', values='Households').reset_index()
df_pums['Percentage'] = df_pums['Households receiving SNAP'] / df_pums['Total households']


df_pums = df_pums[df_pums['Year'].isin([2021, 2023])]
df_pums = df_pums.sort_values('Year', ascending=False)
df_pums = df_pums.reset_index(drop=True)


display(df_pums)


## Exporting ---


if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_pums.to_excel(writer, sheet_name=table_id, index=False)



Chamber_H_19
- Table 6C

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'SUBJECT5'
geographies =  'MPO'
indicator = 'Chamber_H_19'
geography = 'MPO'
table_id = '6C'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_pums = pd.read_excel(file_in, sheet_name=sheet_name)
df_pums = df_pums[['Year', 'Variable', 'Population']]
df_pums = df_pums.pivot_table(index=['Year'], columns='Variable', values='Population').reset_index()

df_pums['English only_Percent'] = df_pums['English only'] / df_pums['Total population']
df_pums['Spanish_Percent'     ] = df_pums['Spanish'     ] / df_pums['Total population']
df_pums['Other_Percent'       ] = df_pums['Other'       ] / df_pums['Total population']
df_pums = df_pums.drop(['Total population', 'English only', 'Spanish', 'Other'], axis=1)

df_pums = df_pums.melt(id_vars='Year', var_name='Language', value_name='Percentage')


df_pums = df_pums.sort_values('Year', ascending=False)
df_pums = df_pums.reset_index(drop=True)

display(df_pums)


## Exporting ---


if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_pums.to_excel(writer, sheet_name=table_id, index=False)



Chamber_H_20
- Table 6D

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS1'
geographies =  'MPO'
indicator = 'Chamber_H_20'
geography = 'MPO'
table_id = '6D'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_acs = pd.read_excel(file_in, sheet_name=sheet_name)
df_acs = df_acs[['Year', 'Race_Ethnicity', 'Variable', 'Population']]
df_acs = df_acs.pivot_table(index=['Year', 'Variable'], columns='Race_Ethnicity', values='Population').reset_index()
df_acs['Not Hispanic or Latino'] = df_acs['All'] - df_acs['Hispanic or Latino']
df_acs = df_acs.melt(id_vars=['Year', 'Variable'], var_name='Race_Ethnicity', value_name='Population')
df_acs = df_acs[df_acs['Race_Ethnicity'] != 'All']
df_acs = df_acs.pivot_table(index=['Year', 'Race_Ethnicity'], columns='Variable', values='Population').reset_index()

if 'ACS' in estimate:
    df_acs['Percentage'] = df_acs['Total Income in the past 12 months below poverty level'] / df_acs['Total']
if 'SUBJECT' in estimate:
    df_acs['Percentage'] = df_acs['Population below the poverty line'] / df_acs['Total population']

df_acs = df_acs[['Year', 'Race_Ethnicity', 'Percentage']]
df_acs = df_acs.pivot_table(index='Year', columns='Race_Ethnicity', values='Percentage').reset_index()

df_acs = df_acs.sort_values('Year', ascending=True)
df_acs = df_acs.reset_index(drop=True)


display(df_acs)



In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'PUMS5'
geographies =  'MPO'
indicator = 'Chamber_H_20'
geography = 'MPO'
table_id = '6D'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_pums0 = pd.read_excel(file_in, sheet_name=sheet_name)
df_pums = df_pums0.copy()

conditions = [
    df_pums['POVPIP'] == -1
    , (df_pums['POVPIP'] >= 0) & (df_pums['POVPIP'] <= 100)
    , df_pums['POVPIP'] >= 101
]

choices = ['Not reported', 'Below the poverty level', 'Above the poverty level']

df_pums['Category'] = np.select(conditions, choices, default='No')


df_pums = df_pums.groupby(['Year', 'HISP', 'Category'], as_index=False).agg(Population=('Population', 'sum'))

df_pums['Percentage'] = df_pums['Population'] / df_pums.groupby(['Year', 'HISP'])['Population'].transform('sum')
df_pums = df_pums[df_pums['Category'] == 'Below the poverty level']
df_pums = df_pums.drop('Category', axis=1)
df_pums = df_pums.reset_index(drop=True)
df_pums = df_pums.pivot_table(index=['Year'], columns='HISP', values='Percentage').reset_index()

display(df_pums)


In [ ]:
df_pums0.groupby('Year', as_index=False)['Population'].sum()

Chamber_H_21
- Table 6E

In [ ]:


path_in  = path_csm / 'original_exports'
estimate = 'ACS5'
geographies =  'MPO'
indicator = 'Chamber_H_21'
geography = 'MPO'
table_id = '6E'


workbook = f'{indicator} {geography} {estimate}.xlsx'
sheet_name = geography

## Import ---

file_in = path_in / workbook
df_acs = pd.read_excel(file_in, sheet_name=sheet_name)
df_acs = df_acs[['Year', 'Race_Ethnicity', 'Variable', 'Population']]
df_acs = df_acs.pivot_table(index=['Year', 'Variable'], columns='Race_Ethnicity', values='Population').reset_index()
df_acs['Not Hispanic or Latino'] = df_acs['All'] - df_acs['Hispanic or Latino']
df_acs = df_acs.melt(id_vars=['Year', 'Variable'], var_name='Race_Ethnicity', value_name='Population')
df_acs = df_acs[df_acs['Race_Ethnicity'] != 'All']
# df_acs = df_acs.pivot_table(index=['Year', 'Race_Ethnicity'], columns='Variable', values='Population').reset_index()
df_acs['Category'] = df_acs['Variable'].copy()
df_acs['Category'] = df_acs['Category'].str.replace(' With health insurance coverage', '')
df_acs['Category'] = df_acs['Category'].str.replace(' No health insurance coverage'  , '')


df_acs['Variable'] = df_acs['Variable'].str.replace('Total Under 19 years '   , '')
df_acs['Variable'] = df_acs['Variable'].str.replace('Total 19 to 64 years '   , '')
df_acs['Variable'] = df_acs['Variable'].str.replace('Total 65 years and over ', '')


df_acs['Percentage'] = df_acs['Population'] / df_acs.groupby(['Year', 'Race_Ethnicity', 'Category'], as_index=False)['Population'].transform('sum')


df_acs2 = df_acs.groupby(['Year', 'Race_Ethnicity', 'Variable'], as_index=False)['Population'].sum()
df_acs2['Percentage'] = df_acs2['Population'] / df_acs2.groupby(['Year', 'Race_Ethnicity'], as_index=False)['Population'].transform('sum')


df_acs  = df_acs [df_acs ['Variable'] != 'With health insurance coverage']
df_acs2 = df_acs2[df_acs2['Variable'] != 'With health insurance coverage']
df_acs2['Category'] = 'All ages'


df_acs = pd.concat([df_acs, df_acs2])

df_acs = df_acs[['Year', 'Category', 'Race_Ethnicity', 'Percentage']]
df_acs = df_acs.pivot_table(index=['Year', 'Category'], columns='Race_Ethnicity', values='Percentage').reset_index()

df_acs['Sort'] = pd.Categorical(df_acs['Category'], [
    'Total Under 19 years'
    , 'Total 19 to 64 years'
    , 'Total 65 years and over'
    , 'All ages'
])

df_acs = df_acs.sort_values(['Year', 'Sort'], ascending=[False, True])
df_acs = df_acs.drop('Sort', axis=1)
df_acs = df_acs.reset_index(drop=True)


display(df_acs)


## Exporting ---


if export:
    workbook = f'post2.xlsx'
    file_out = path_csm / 'post' / workbook

    with pd.ExcelWriter(file_out,mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_acs.to_excel(writer, sheet_name=table_id, index=False)

